# UD4.04 — Plotly: gráficos que el lector puede tocar

**Módulo 5073 · Programación de Inteligencia Artificial · Curso 2026/27**
UD4 — Visualización de datos · 14 horas

Criterio 1.d · Material de partida de la práctica P4.1

## Objetivos de Aprendizaje

Al finalizar este notebook, serás capaz de:

- **Explicar qué aporta la interactividad** y qué cuesta, con tamaños medidos
- **Usar Plotly Express** para los gráficos habituales, y **Graph Objects** cuando
  hace falta control
- **Personalizar el `hover`**, que es la interacción que más rendimiento da
- **Construir animaciones** y saber cuándo son información y cuándo espectáculo
- **Dibujar en tres dimensiones** de verdad, que es donde Plotly gana a Matplotlib
- **Elegir entre `Scatter` y `Scattergl`** según el número de puntos, midiéndolo
- **Decidir la biblioteca** con un argumento numérico, que es el criterio 1.d

## 1. Qué es Plotly, y en qué se diferencia

Matplotlib y Seaborn producen **una imagen**: una rejilla de píxeles, o un conjunto de
instrucciones de dibujo en un PDF. Plotly produce **una página web**: un fragmento de
HTML con los datos en JSON y una biblioteca de JavaScript que los dibuja en el
navegador y responde al ratón.

De esa diferencia sale todo lo demás:

| | Matplotlib / Seaborn | Plotly |
|---|---|---|
| Qué produce | Una imagen (PNG, PDF, SVG) | HTML + JSON + JavaScript |
| Dónde se dibuja | En Python, una vez | En el navegador, cada vez que el lector toca algo |
| Interactividad | Ninguna | Zoom, desplazamiento, `hover`, filtrar en la leyenda |
| Dónde funciona | En cualquier sitio, incluso en papel | Solo donde hay un navegador |
| Los datos | Se quedan en Python | **Van dentro del fichero** |

La última fila es la que casi nadie tiene en cuenta y tiene dos consecuencias:

1. **El fichero pesa lo que pesen los datos.** Un PNG de un millón de puntos pesa lo
   mismo que uno de diez. Un HTML de Plotly con un millón de puntos lleva el millón de
   puntos dentro. Se mide en la sección 7.
2. **Publicar el gráfico es publicar los datos.** Si el gráfico sale de una tabla con
   datos personales, ese HTML los contiene, aunque no se vean. Es un problema de
   protección de datos, no de rendimiento.

In [ ]:
import io
import time

import numpy as np
import pandas as pd
import plotly
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

rng = np.random.default_rng(20262027)

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

# La plantilla se fija una vez, como el `set_theme` de Seaborn. `plotly_white` es la
# equivalente a `whitegrid`: fondo blanco y rejilla suave.
pio.templates.default = "plotly_white"

# OJO: NO se toca `pio.renderers.default`. Plotly detecta solo dónde se está
# ejecutando —Jupyter, Colab, VS Code— y elige el modo adecuado. Ponerlo a mano, y en
# particular ponerlo a "browser", es la causa de que los gráficos no salgan en el
# cuaderno y se abran en una pestaña aparte.

print("plotly", plotly.__version__)
print("pandas", pd.__version__)

## 2. Los datos

Plotly trae conjuntos de ejemplo y, a diferencia de los de Seaborn, **vienen dentro
del paquete**: no hacen falta ni red ni descarga. Usamos dos:

- **`gapminder`**: esperanza de vida, renta por habitante y población de 142 países,
  cada cinco años de 1952 a 2007. Es el conjunto con el que Hans Rosling hizo célebre
  la visualización animada, y sigue siendo el mejor ejemplo que existe.
- **`stocks`**: precios normalizados de seis empresas tecnológicas, para las series
  temporales.

Y los datos del módulo, TechStore, para lo que tiene que ver con la práctica.

In [ ]:
gapminder = px.data.gapminder()
acciones = px.data.stocks()

print(f"gapminder: {gapminder.shape[0]} filas, "
      f"{gapminder['country'].nunique()} países, "
      f"{gapminder['year'].min()}-{gapminder['year'].max()}")
print(f"acciones:  {acciones.shape[0]} fechas, "
      f"{acciones.shape[1] - 1} empresas")
print()
print(gapminder.head())

In [ ]:
# Los datos de TechStore, limpios igual que en el cuaderno 03.
import os
import urllib.request

BASE = ("https://raw.githubusercontent.com/RafaSalaEsteve/IABD-PIA-notebooks"
        "/main/UD4/datos/")
FICHERO = "ecommerce_ventas_2024.csv"

if not os.path.exists(os.path.join("datos", FICHERO)):
    os.makedirs("datos", exist_ok=True)
    urllib.request.urlretrieve(BASE + FICHERO, os.path.join("datos", FICHERO))
    print("descargado:", FICHERO)

ventas = pd.read_csv(os.path.join("datos", FICHERO))
ventas["Fecha"] = pd.to_datetime(ventas["Fecha"])
ventas["Categoria"] = (ventas["Categoria"].str.strip().str.lower()
                       .str.normalize("NFKD")
                       .str.encode("ascii", "ignore").str.decode("ascii"))
ventas = ventas[(ventas["Precio_Unitario"] > 0)
                & ventas["Cantidad"].between(1, 50)
                & (ventas["Descuento_%"].between(0, 100)
                   | ventas["Descuento_%"].isna())
                & ventas["Region"].notna()].copy()
ventas["Descuento_%"] = ventas["Descuento_%"].fillna(0.0)
ventas["Importe"] = (ventas["Precio_Unitario"] * ventas["Cantidad"]
                     * (1 - ventas["Descuento_%"] / 100) + ventas["Costo_Envio"])
ventas["Mes"] = ventas["Fecha"].dt.to_period("M").dt.to_timestamp()
ventas = ventas.reset_index(drop=True)

print(f"TechStore limpio: {len(ventas)} transacciones, "
      f"{ventas['Categoria'].nunique()} categorías, "
      f"{ventas['Region'].nunique()} regiones")

## 3. Plotly Express

`plotly.express` es la capa de alto nivel, y su parecido con Seaborn no es casual:
recibe un `DataFrame`, los nombres de las columnas y las variables que van en el color,
el tamaño y los paneles.

La diferencia con Seaborn: **`px` devuelve un objeto `Figure` que se puede seguir
modificando** con `update_layout`, `update_traces` y `add_trace`. Eso es lo que
sustituye a «volver a Matplotlib para rematar».

In [ ]:
fig = px.scatter(
    ventas.sample(1200, random_state=20262027),
    x="Precio_Unitario",
    y="Importe",
    color="Categoria",
    size="Cantidad",
    size_max=22,
    hover_data=["Region", "Producto", "Descuento_%"],
    title="Precio unitario e importe por transacción",
    labels={"Precio_Unitario": "Precio unitario (€)",
            "Importe": "Importe total (€)",
            "Cantidad": "Unidades",
            "Categoria": "Categoría"},
    height=560,
)
fig.update_traces(marker=dict(line=dict(width=0.5, color="white")))
fig.show()

print("Lo que se puede hacer con este gráfico y no con el mismo en Matplotlib:")
print()
print("  - Pasar el ratón por un punto y leer sus valores exactos, más las tres")
print("    columnas de `hover_data`, sin saturar el gráfico de etiquetas.")
print("  - Arrastrar un rectángulo para ampliar una zona, y doble clic para volver.")
print("  - Hacer clic en una categoría de la leyenda para ocultarla. Con seis")
print("    categorías superpuestas, esto es la diferencia entre ver algo y no verlo.")
print("  - Doble clic en una categoría de la leyenda para ver SOLO esa.")
print()
print("Ese último punto es el argumento de fondo: en un gráfico estático, seis series")
print("solapadas obligan a dibujar seis paneles. En uno interactivo, no.")

### 3.1 Series temporales, y el selector de rango

En series temporales largas la interactividad deja de ser una comodidad. Un gráfico
estático de cinco años de datos diarios no permite mirar una semana concreta: hay que
dibujar otro gráfico. Con el selector de rango y la barra deslizante, el lector lo
hace solo.

In [ ]:
fig = px.line(
    acciones,
    x="date",
    y=acciones.columns[1:],
    title="Precio normalizado de seis tecnológicas (base 1 en enero de 2018)",
    labels={"value": "Precio normalizado", "variable": "Empresa",
            "date": "Fecha"},
    height=520,
)

fig.update_xaxes(
    rangeslider_visible=True,
    rangeselector=dict(buttons=[
        dict(count=1, label="1 mes", step="month", stepmode="backward"),
        dict(count=6, label="6 meses", step="month", stepmode="backward"),
        dict(count=1, label="año actual", step="year", stepmode="todate"),
        dict(count=1, label="1 año", step="year", stepmode="backward"),
        dict(step="all", label="Todo"),
    ]),
)
# `hovermode="x unified"` muestra las SEIS series a la vez en la fecha señalada, en
# lugar de solo la más cercana. Para comparar es lo que se quiere casi siempre.
fig.update_layout(hovermode="x unified")
fig.show()

print("Dos ajustes que valen su peso en oro en series temporales:")
print()
print("  rangeslider_visible=True   la barra de abajo para elegir el tramo")
print("  hovermode='x unified'      las seis series a la vez en la fecha señalada")
print()
print("Sin el segundo, al pasar el ratón sale solo la serie más cercana, que casi")
print("nunca es lo que se quiere cuando el gráfico está para comparar.")

### 3.2 Los gráficos estadísticos, y el margen

`px` tiene los mismos tipos que Seaborn, con un añadido propio: el parámetro
`marginal`, que añade un resumen del reparto en el borde. Es el `jointplot` de Seaborn
sin tener que cambiar de nivel de función.

In [ ]:
fig = px.histogram(
    ventas[ventas["Importe"] < ventas["Importe"].quantile(0.99)],
    x="Importe",
    color="Categoria",
    marginal="box",
    nbins=45,
    opacity=0.65,
    barmode="overlay",
    title="Reparto del importe por categoría, con la caja al margen",
    labels={"Importe": "Importe (€)", "Categoria": "Categoría",
            "count": "Transacciones"},
    height=580,
)
fig.show()

fig = px.box(
    ventas,
    x="Categoria",
    y="Importe",
    color="Categoria",
    points="outliers",
    title="Importe por categoría · el hover da los cinco números de cada caja",
    labels={"Importe": "Importe (€)", "Categoria": "Categoría"},
    height=520,
)
fig.update_yaxes(range=[0, float(ventas["Importe"].quantile(0.98))])
fig.update_layout(showlegend=False)
fig.show()

print("La ventaja concreta de la caja interactiva: al pasar el ratón salen el mínimo,")
print("los tres cuartiles y el máximo con sus valores exactos.")
print()
print("En una caja de Matplotlib esos cinco números no están: hay que calcularlos")
print("aparte y ponerlos en una tabla al lado. Aquí ya están dentro del gráfico.")

## 4. El `hover`, que es la interacción que más aporta

De todo lo que hace Plotly, el `hover` es lo que más cambia la utilidad de un gráfico,
y es lo que menos se personaliza.

El motivo es de fondo: en un gráfico estático hay que **elegir** entre poner las
etiquetas (y saturarlo) o no ponerlas (y perder el detalle). El `hover` rompe esa
disyuntiva: el gráfico queda limpio y el detalle está a un gesto.

Se controla con `hovertemplate`, que es una cadena con marcadores:

| Marcador | Es |
|---|---|
| `%{x}`, `%{y}` | Los valores de los ejes |
| `%{customdata[0]}` | La primera columna extra que le pases |
| `%{marker.size}` | El tamaño del punto |
| `:.2f`, `:,.0f`, `:.1%` | El formato del número |
| `<b>`, `<br>` | Negrita y salto de línea |
| `<extra></extra>` | **Quita** la cajita gris de la derecha |

In [ ]:
muestra = ventas.sample(700, random_state=20262027)

# Sin personalizar: Plotly pone los nombres de las columnas tal cual.
fig = px.scatter(muestra, x="Precio_Unitario", y="Importe", color="Categoria",
                 title="Hover por defecto", height=380)
fig.show()

# Personalizado: un texto pensado para quien lo va a leer.
fig = go.Figure()
for categoria, grupo in muestra.groupby("Categoria"):
    fig.add_trace(go.Scatter(
        x=grupo["Precio_Unitario"],
        y=grupo["Importe"],
        mode="markers",
        name=categoria,
        marker=dict(size=8, opacity=0.75,
                    line=dict(width=0.5, color="white")),
        # `customdata` lleva las columnas que no están en los ejes y que se quieren
        # poder mostrar. Es un array de (n_puntos, n_columnas).
        customdata=grupo[["Producto", "Region", "Cantidad",
                          "Descuento_%"]].to_numpy(),
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "Región: %{customdata[1]}<br>"
            "<br>"
            "Precio unitario: %{x:,.2f} €<br>"
            "Unidades: %{customdata[2]:.0f}<br>"
            "Descuento: %{customdata[3]:.0f} %<br>"
            "<b>Importe: %{y:,.2f} €</b>"
            "<extra></extra>"
        ),
    ))

fig.update_layout(
    title="Hover personalizado · el mismo gráfico, otra utilidad",
    xaxis_title="Precio unitario (€)",
    yaxis_title="Importe (€)",
    legend_title="Categoría",
    height=520,
)
fig.show()

print("Compara los dos. Los datos son los mismos y el gráfico es el mismo.")
print()
print("En el primero, el hover dice 'Precio_Unitario=63.44'. En el segundo dice qué")
print("producto es, de qué región, cuántas unidades, cuánto descuento y el importe")
print("con separador de miles y símbolo de euro.")
print()
print("Un cuadro de mando que va a mirar alguien que no ha escrito el código se")
print("juega su utilidad en esa diferencia.")

## 5. Animaciones

`animation_frame` reparte los datos en fotogramas por el valor de una columna, y
Plotly añade el botón de reproducción y la barra deslizante.

Dos parámetros hacen la diferencia entre una animación que informa y una que marea:

- **`animation_group`** mantiene la identidad de cada elemento entre fotogramas. Sin
  él, los puntos no se corresponden entre un año y el siguiente y el movimiento es un
  parpadeo. Con él, cada país tiene una trayectoria que se puede seguir.
- **`range_x` y `range_y` fijos.** Sin fijarlos, los ejes se reescalan en cada
  fotograma y el movimiento que se ve es el de los ejes, no el de los datos. Es el
  error que arruina la mayoría de las animaciones que se ven por ahí.

In [ ]:
fig = px.scatter(
    gapminder,
    x="gdpPercap",
    y="lifeExp",
    size="pop",
    color="continent",
    hover_name="country",
    animation_frame="year",
    animation_group="country",   # cada país conserva su identidad entre fotogramas
    log_x=True,
    size_max=55,
    range_x=[100, 100_000],      # rangos FIJOS: si no, se mueven los ejes
    range_y=[25, 90],
    title="Salud y riqueza, 1952-2007 · el gráfico de Hans Rosling",
    labels={"gdpPercap": "Renta por habitante (dólares, escala logarítmica)",
            "lifeExp": "Esperanza de vida (años)",
            "pop": "Población", "continent": "Continente", "year": "Año"},
    height=620,
)
fig.show()

print("Este gráfico tiene cinco variables a la vez: renta (X), esperanza de vida (Y),")
print("población (tamaño), continente (color) y año (tiempo). Cinco.")
print()
print("Un gráfico estático llega a cuatro con esfuerzo. La quinta —el tiempo— es la")
print("que la animación aporta y ninguna imagen puede dar.")
print()
print("Y la advertencia: es el ÚNICO caso en el que el tiempo animado gana a doce")
print("paneles pequeños. Si lo que hay que hacer es COMPARAR dos momentos, dos")
print("paneles lado a lado ganan siempre, porque el ojo no puede comparar con la")
print("memoria de hace tres segundos.")

In [ ]:
# La comparación honesta: los dos extremos de la animación, en paneles.
extremos = gapminder[gapminder["year"].isin([1952, 2007])]

fig = px.scatter(
    extremos, x="gdpPercap", y="lifeExp", size="pop", color="continent",
    hover_name="country", facet_col="year", log_x=True, size_max=45,
    range_y=[25, 90],
    title="Los dos extremos, lado a lado: ahora sí se pueden comparar",
    labels={"gdpPercap": "Renta por habitante (dólares)",
            "lifeExp": "Esperanza de vida (años)", "continent": "Continente"},
    height=470,
)
fig.show()

resumen = gapminder[gapminder["year"].isin([1952, 2007])].groupby("year").agg(
    vida_media=("lifeExp", "mean"), renta_mediana=("gdpPercap", "median"))
print(resumen.round(1).to_string())
print()
print("Con los dos paneles, en dos segundos se ve lo que la animación tarda quince")
print("en contar: todo el mundo se ha movido arriba y a la derecha, y la nube se ha")
print("juntado.")
print()
print("Norma práctica: la animación es para EXPLORAR o para CONTAR una historia en")
print("directo. Para un informe que alguien lee solo, paneles.")

## 6. Tres dimensiones: aquí Plotly gana

En el cuaderno 02 quedó dicho que un 3D estático suele ser mala idea, porque el lector
no puede girarlo y sin girarlo no juzga la profundidad. **Plotly resuelve exactamente
esa objeción**: el gráfico se gira con el ratón.

Es la única categoría en la que la comparación no está reñida.

In [ ]:
iris = px.data.iris()

fig = px.scatter_3d(
    iris,
    x="sepal_length", y="sepal_width", z="petal_width",
    color="species", size="petal_length", size_max=18, opacity=0.8,
    title="Iris en tres dimensiones · gíralo con el ratón",
    labels={"sepal_length": "Largo del sépalo (cm)",
            "sepal_width": "Ancho del sépalo (cm)",
            "petal_width": "Ancho del pétalo (cm)",
            "species": "Especie"},
    height=640,
)
fig.update_layout(scene=dict(camera=dict(eye=dict(x=1.6, y=1.6, z=1.1))))
fig.show()

print("Gíralo y busca el ángulo desde el que las tres especies se separan mejor.")
print("Ese gesto —buscar el ángulo— es literalmente lo que un 3D estático no permite,")
print("y es la razón por la que en el cuaderno 02 se recomendaba el contorno.")
print()
print("Y una vez encontrado el ángulo bueno, la pregunta útil es: ¿hacen falta las")
print("tres dimensiones? Mira si dos de las tres variables ya separan las especies.")
print("Si es así, una nube de puntos plana es mejor gráfico, porque se puede imprimir.")

In [ ]:
# La superficie 3D, que es el otro caso donde el 3D interactivo es la respuesta.
malla = np.linspace(-5, 5, 70)
X, Y = np.meshgrid(malla, malla)
Z = np.sin(np.sqrt(X ** 2 + Y ** 2))

fig = go.Figure(data=[go.Surface(x=X, y=Y, z=Z, colorscale="Viridis",
                                 colorbar=dict(title="z"))])
fig.update_layout(
    title="z = sin(√(x² + y²)) · una superficie que hay que girar para entender",
    scene=dict(xaxis_title="x", yaxis_title="y", zaxis_title="z",
               camera=dict(eye=dict(x=1.6, y=1.6, z=1.0))),
    height=640,
)
fig.show()

## 7. Lo que cuesta la interactividad, medido

Esta es la sección que responde al criterio 1.d, y es la que convierte «Plotly pesa
más» en un número que se puede citar.

Recuerda de la sección 1: **el HTML de Plotly lleva los datos dentro**. Vamos a medir
cuánto ocupa el mismo gráfico según el número de puntos, y a compararlo con el PNG
equivalente.

In [ ]:
def tamano_html(figura, incluir_biblioteca=False):
    """Tamaño en kilobytes del HTML de una figura.

    Con `incluir_biblioteca=False` se mide solo el gráfico —datos y configuración—,
    que es lo que crece con el número de puntos. La biblioteca de JavaScript son
    unos 3 MB fijos que se pueden cargar desde un CDN una sola vez.
    """
    html = figura.to_html(include_plotlyjs="cdn" if not incluir_biblioteca else True,
                          full_html=False)
    return len(html.encode("utf-8")) / 1024


import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt


def tamano_png(x, y, dpi=150):
    """Tamaño en kilobytes del PNG del mismo gráfico hecho con Matplotlib."""
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.plot(x, y, "o", markersize=2, alpha=0.3)
    memoria = io.BytesIO()
    fig.savefig(memoria, format="png", dpi=dpi, bbox_inches="tight")
    plt.close(fig)
    return len(memoria.getvalue()) / 1024


print(f"{'puntos':>10}  {'HTML Plotly':>13}  {'PNG matplotlib':>16}  "
      f"{'proporción':>11}")
print("-" * 58)

medidas = []
for n in (100, 1_000, 10_000, 100_000):
    x, y = rng.normal(size=n), rng.normal(size=n)
    figura = go.Figure(go.Scatter(x=x, y=y, mode="markers",
                                  marker=dict(size=3, opacity=0.4)))
    kb_html = tamano_html(figura)
    kb_png = tamano_png(x, y)
    medidas.append((n, kb_html, kb_png))
    print(f"{n:>10,}  {kb_html:>10,.0f} KB  {kb_png:>13,.0f} KB  "
          f"{kb_html / kb_png:>10.1f}×")

print()
print("Y ahora los dos números que hay que recordar:")
print()
print(f"  La biblioteca de JavaScript de Plotly, incrustada en el fichero, ocupa")
figura_minima = go.Figure(go.Scatter(x=[1, 2], y=[1, 2]))
print(f"  {tamano_html(figura_minima, incluir_biblioteca=True):,.0f} KB. "
      f"Con include_plotlyjs='cdn' se queda en "
      f"{tamano_html(figura_minima):,.0f} KB")
print("  y se carga de internet, pero entonces el fichero NO funciona sin conexión.")
print()
print("  El PNG pesa lo mismo tenga cien puntos o cien mil, porque solo guarda")
print("  píxeles. El HTML crece linealmente con los datos, porque los lleva dentro.")

### 7.1 `Scatter` y `Scattergl`

Con muchos puntos el problema no es solo el tamaño: es que el navegador tiene que
dibujarlos. `go.Scatter` usa SVG, o sea que crea **un elemento del documento por
punto**, y el navegador se atasca por encima de unos pocos miles.

`go.Scattergl` usa WebGL y dibuja con la tarjeta gráfica. Aguanta cientos de miles de
puntos sin problema. La contrapartida: algunas opciones de aspecto no están, y en
equipos sin aceleración gráfica puede no funcionar.

In [ ]:
print(f"{'puntos':>10}  {'Scatter (SVG)':>15}  {'Scattergl (WebGL)':>19}  "
      f"{'ahorro':>8}")
print("-" * 60)

for n in (1_000, 10_000, 100_000):
    x, y = rng.normal(size=n), rng.normal(size=n)
    svg = go.Figure(go.Scatter(x=x, y=y, mode="markers", marker=dict(size=3)))
    webgl = go.Figure(go.Scattergl(x=x, y=y, mode="markers", marker=dict(size=3)))
    kb_svg, kb_webgl = tamano_html(svg), tamano_html(webgl)
    print(f"{n:>10,}  {kb_svg:>12,.0f} KB  {kb_webgl:>16,.0f} KB  "
          f"{(1 - kb_webgl / kb_svg) * 100:>7.0f} %")

print()
print("El tamaño del fichero es parecido —los datos son los mismos—, y la diferencia")
print("está en cómo los dibuja el navegador. Eso no se puede medir desde Python:")
print("hay que abrir los dos y moverlos.")
print()
print("La regla, y es sencilla:")
print()
print("  hasta ~5.000 puntos    go.Scatter, o px.scatter sin más")
print("  de 5.000 a ~500.000    go.Scattergl, o px.scatter(render_mode='webgl')")
print("  más de 500.000         AGREGAR antes de dibujar, como en el cuaderno 02")

In [ ]:
# La tercera vía, que es la misma del cuaderno 02: agregar en lugar de dibujar.
n = 300_000
x, y = rng.normal(size=n), rng.normal(size=n)

todos = go.Figure(go.Scattergl(x=x, y=y, mode="markers",
                               marker=dict(size=2, opacity=0.25)))
todos.update_layout(title=f"{n:,} puntos con Scattergl", height=420,
                    xaxis_title="x", yaxis_title="y")

agregado = px.density_heatmap(x=x, y=y, nbinsx=90, nbinsy=90,
                              color_continuous_scale="Viridis")
agregado.update_layout(
    title=f"Los mismos {n:,} puntos agregados en una rejilla de 90×90",
    height=420, xaxis_title="x", yaxis_title="y")

print(f"{'estrategia':>34}  {'HTML':>10}")
print("-" * 48)
print(f"{'Scattergl con todos los puntos':>34}  {tamano_html(todos):>7,.0f} KB")
print(f"{'density_heatmap (agregado)':>34}  {tamano_html(agregado):>7,.0f} KB")
print()
print(f"Reducción: {(1 - tamano_html(agregado) / tamano_html(todos)) * 100:.1f} %")
print()
print("Y además se lee mejor, por lo mismo que en el cuaderno 02: 300.000 puntos")
print("superpuestos son una mancha, y el recuento por celda no.")

todos.show()
agregado.show()

## 8. Composición y ejes secundarios

`make_subplots` es el equivalente de `plt.subplots`, con una diferencia: hay que
declarar el **tipo** de cada panel en `specs`, porque un panel 3D o un mapa no se
gestionan igual que uno plano.

In [ ]:
resumen_mes = ventas.groupby("Mes", as_index=False).agg(
    importe=("Importe", "sum"), transacciones=("Importe", "size"))
resumen_cat = (ventas.groupby("Categoria", as_index=False)["Importe"].sum()
               .sort_values("Importe"))

fig = make_subplots(
    rows=2, cols=2,
    subplot_titles=("Importe total por mes", "Importe por categoría",
                    "Reparto del importe", "Importe medio por región"),
    specs=[[{"secondary_y": True}, {"type": "bar"}],
           [{"type": "histogram"}, {"type": "bar"}]],
    vertical_spacing=0.16,
)

# Panel 1: el caso del eje secundario, con la misma advertencia del cuaderno 02.
fig.add_trace(go.Bar(x=resumen_mes["Mes"], y=resumen_mes["importe"],
                     name="Importe (€)", marker_color="#5dade2"),
              row=1, col=1, secondary_y=False)
fig.add_trace(go.Scatter(x=resumen_mes["Mes"], y=resumen_mes["transacciones"],
                         name="Transacciones", mode="lines+markers",
                         line=dict(color="#c0392b", width=3)),
              row=1, col=1, secondary_y=True)

fig.add_trace(go.Bar(x=resumen_cat["Importe"], y=resumen_cat["Categoria"],
                     orientation="h", name="Por categoría",
                     marker_color="#48c9b0", showlegend=False),
              row=1, col=2)

fig.add_trace(go.Histogram(x=ventas.loc[ventas["Importe"] <
                                        ventas["Importe"].quantile(0.99),
                                        "Importe"],
                           nbinsx=45, name="Importe", marker_color="#af7ac5",
                           showlegend=False),
              row=2, col=1)

resumen_region = (ventas.groupby("Region", as_index=False)["Importe"].mean()
                  .sort_values("Importe", ascending=False))
fig.add_trace(go.Bar(x=resumen_region["Region"], y=resumen_region["Importe"],
                     name="Por región", marker_color="#f5b041",
                     showlegend=False),
              row=2, col=2)

fig.update_yaxes(title_text="Importe (€)", row=1, col=1, secondary_y=False)
fig.update_yaxes(title_text="Transacciones", row=1, col=1, secondary_y=True)
fig.update_xaxes(title_text="Importe total (€)", row=1, col=2)
fig.update_xaxes(title_text="Importe (€)", row=2, col=1)
fig.update_yaxes(title_text="Transacciones", row=2, col=1)
fig.update_yaxes(title_text="Importe medio (€)", row=2, col=2)
fig.update_xaxes(tickangle=45, row=2, col=2)

fig.update_layout(
    title_text="TechStore 2024 · cuadro de mando en cuatro paneles",
    height=760, hovermode="closest",
    legend=dict(orientation="h", yanchor="bottom", y=1.06, x=0),
)
fig.show()

correlacion = float(resumen_mes[["importe", "transacciones"]].corr().iloc[0, 1])
print(f"En el primer panel hay dos escalas. Correlación real entre importe y")
print(f"número de transacciones por mes: {correlacion:+.3f}")
print()
print("Aquí el doble eje SÍ está justificado, y la razón es que la correlación es")
print(f"de {correlacion:.2f}: las dos series miden casi lo mismo, y el gráfico lo que")
print("hace es enseñar que el importe sube porque hay más transacciones y no porque")
print("suba el ticket medio.")
print()
print("La advertencia del cuaderno 02 sigue en pie: con dos series que NO están")
print("relacionadas, el doble eje fabrica la relación. La diferencia es que aquí la")
print("correlación está medida y escrita.")

## 9. Exportar

Cuatro salidas, y cada una para una cosa distinta:

| Método | Produce | Para qué |
|---|---|---|
| `write_html(ruta)` | HTML interactivo | Enviarlo, subirlo a Aules, publicarlo |
| `to_html(full_html=False)` | Un fragmento | Incrustarlo en una página o en Streamlit |
| `write_image(ruta)` | PNG, PDF, SVG | Un informe impreso. **Necesita `kaleido`** |
| `to_json()` | La especificación en JSON | Guardar el gráfico, reconstruirlo, una API |

Y el detalle de `write_html` que decide si el fichero sirve:

- `include_plotlyjs=True`: mete los ~3 MB de JavaScript dentro. El fichero pesa, y
  **funciona sin conexión**.
- `include_plotlyjs="cdn"`: los carga de internet. El fichero es pequeño y **no
  funciona sin conexión**.

Para entregar una práctica que el profesorado tiene que poder abrir, `True`.

In [ ]:
figura_ejemplo = px.scatter(
    ventas.sample(600, random_state=20262027),
    x="Precio_Unitario", y="Importe", color="Categoria",
    title="Figura de ejemplo para exportar",
    labels={"Precio_Unitario": "Precio unitario (€)", "Importe": "Importe (€)"},
    height=430)

print(f"{'salida':>44}  {'tamaño':>10}")
print("-" * 58)
print(f"{'HTML con la biblioteca dentro (include_plotlyjs=True)':>44}  "
      f"{tamano_html(figura_ejemplo, incluir_biblioteca=True):>7,.0f} KB")
print(f"{'HTML apoyado en el CDN (include_plotlyjs=\"cdn\")':>44}  "
      f"{tamano_html(figura_ejemplo):>7,.0f} KB")
print(f"{'JSON de la especificación':>44}  "
      f"{len(figura_ejemplo.to_json().encode('utf-8')) / 1024:>7,.0f} KB")

# La imagen estática necesita `kaleido`, que es un paquete aparte y a veces no está.
# Se envuelve en un try para que el cuaderno no se rompa donde no esté instalado: es
# la misma guarda que en la UD2 con las claves de Azure.
try:
    memoria = io.BytesIO()
    figura_ejemplo.write_image(memoria, format="png", width=1100, height=600, scale=2)
    print(f"{'PNG con kaleido (1100x600, escala 2)':>44}  "
          f"{len(memoria.getvalue()) / 1024:>7,.0f} KB")
except Exception as error:
    print(f"{'PNG con kaleido':>44}  no disponible")
    print(f"     ({type(error).__name__}: instala kaleido con `pip install kaleido`)")

print()
print("Lo que hay que sacar de aquí: un gráfico interactivo autocontenido son")
print("megabytes, y casi todo es la biblioteca de JavaScript. Si hay que entregar")
print("VEINTE gráficos, veinte ficheros de tres megas son sesenta megas, y ahí ya")
print("compensa una página con los veinte gráficos y una sola copia de la biblioteca.")

## 10. La decisión, que es el criterio 1.d

Con lo medido en los tres cuadernos ya se puede contestar a la pregunta de verdad, y
no con impresiones.

### Qué elegir, y por qué

| Situación | Herramienta | El argumento medido |
|---|---|---|
| Informe impreso o PDF | **Matplotlib** | El PDF vectorial pesa menos que el PNG y no se pixela (cuaderno 01) |
| Exploración de datos propios | **Seaborn** | La misma figura en cinco líneas en vez de veinte (cuaderno 03) |
| Muchas series solapadas | **Plotly** | Filtrar en la leyenda evita dibujar seis paneles |
| Series temporales largas | **Plotly** | El selector de rango sustituye a un gráfico por tramo |
| Hay que girar una superficie | **Plotly** | Es la objeción del 3D estático, resuelta (cuaderno 02) |
| Cuadro de mando para alguien que no programa | **Plotly** | El `hover` da el detalle sin saturar el gráfico |
| Más de 500.000 puntos | **Ninguna: agregar** | `hexbin` o `density_heatmap` reducen el fichero un 99 % |
| Sin conexión garantizada | **Matplotlib** | Un PNG siempre se abre |
| Los datos son personales | **Matplotlib** | El HTML de Plotly lleva los datos dentro |

### Y la respuesta de fondo

La pregunta del criterio 1.d es qué hace a un lenguaje adecuado para la inteligencia
artificial. En la UD3 la respuesta fue que **Python delega**: el bucle ocurre en C y
Python solo da la orden.

Aquí la respuesta es la otra mitad, y es el **ecosistema**. Estas tres bibliotecas no
compiten: se apilan. Seaborn está escrita sobre Matplotlib; las tres reciben
`DataFrame` de Pandas, que están construidos sobre arrays de NumPy; y `df.plot()` de
Pandas es Matplotlib por dentro. Un `DataFrame` pasa de una a otra sin convertir nada.

Eso es lo que ningún lenguaje más rápido que Python tiene, y es lo que hace que la
comparación no sea entre lenguajes sino entre ecosistemas. Se puede escribir código
numérico más rápido en C++ o en Rust; lo que no se puede es cargar un CSV, limpiarlo,
mirarlo de nueve formas distintas y publicar un cuadro de mando en una tarde.

In [ ]:
# La prueba de que se apilan: los cuatro caminos hasta el mismo gráfico.
figura, ejes = plt.subplots(2, 2, figsize=(13, 8))
figura.suptitle("El mismo histograma por cuatro caminos del mismo ecosistema",
                fontsize=14, fontweight="bold")

recorte = ventas[ventas["Importe"] < ventas["Importe"].quantile(0.99)]

# 1. Matplotlib sobre un array de NumPy.
ejes[0, 0].hist(recorte["Importe"].to_numpy(), bins=40, color="#5dade2",
                edgecolor="#1a5276")
ejes[0, 0].set_title("1. Matplotlib sobre un array de NumPy", fontweight="bold",
                     fontsize=10)

# 2. Pandas, que llama a Matplotlib por dentro.
recorte["Importe"].plot.hist(bins=40, ax=ejes[0, 1], color="#48c9b0",
                             edgecolor="#0e6655")
ejes[0, 1].set_title("2. df.plot.hist(), que es Matplotlib por dentro",
                     fontweight="bold", fontsize=10)

# 3. Seaborn, que también llama a Matplotlib.
import seaborn as sns
sns.histplot(data=recorte, x="Importe", bins=40, ax=ejes[1, 0], color="#af7ac5")
ejes[1, 0].set_title("3. Seaborn, escrita sobre Matplotlib", fontweight="bold",
                     fontsize=10)

# 4. El cuarto no cabe aquí, porque Plotly no dibuja en un Axes de Matplotlib.
ejes[1, 1].axis("off")
ejes[1, 1].text(0.5, 0.5,
                "4. Plotly\n\n"
                "No cabe en este panel, y eso ES el dato:\n"
                "Plotly no dibuja sobre un Axes de Matplotlib\n"
                "porque no está construida encima.\n\n"
                "Produce HTML, no una imagen.\n"
                "Por eso es la única de las cuatro que\n"
                "no se puede mezclar con las otras tres.",
                ha="center", va="center", fontsize=10,
                bbox=dict(boxstyle="round,pad=0.8", facecolor="#fdebd0",
                          edgecolor="#b9770e"))

for eje in ejes.ravel()[:3]:
    eje.set_xlabel("Importe (€)")
    eje.set_ylabel("Transacciones")

figura.tight_layout()
plt.show()

# El cuarto camino, en su propia figura.
px.histogram(recorte, x="Importe", nbins=40,
             title="4. Plotly, que produce HTML y no una imagen",
             labels={"Importe": "Importe (€)", "count": "Transacciones"},
             height=380).show()

print("Los tres primeros paneles son literalmente el mismo dibujo hecho por la misma")
print("biblioteca a través de tres interfaces distintas. El cuarto es otro programa.")

## Ejercicios

Las condiciones de los cuadernos anteriores, más una propia de Plotly:

5. **Todo gráfico interactivo que entregues lleva `hovertemplate` personalizado.** El
   `hover` por defecto, con los nombres de las columnas en crudo, no cuenta como
   entregado.

### Ejercicio 1 (Básico): De Seaborn a Plotly

Coge tres gráficos que hicieras en el cuaderno 03 con Seaborn y rehazlos con
`plotly.express`. Para cada uno, escribe una línea con **qué gana y qué pierde** al
pasarlo a interactivo.

Uno de los tres tiene que ser uno en el que Plotly **pierda**. Encuéntralo.

In [ ]:
# TODO: Escribe tu código aquí

### Ejercicio 2 (Básico): Un `hover` que sirva

Dibuja el importe por transacción de TechStore frente a la fecha, con el color por
región, y personaliza el `hover` para que muestre: el producto en negrita, la
categoría, la región, las unidades, el descuento en porcentaje y el importe con
separador de miles y símbolo de euro. Sin la cajita gris de la derecha.

Después enséñaselo a alguien que no haya visto los datos y comprueba si entiende
cada campo sin preguntar.

In [ ]:
# TODO: Escribe tu código aquí

### Ejercicio 3 (Intermedio): La animación mal hecha y bien hecha

Con `gapminder`, dibuja la misma animación **tres veces**:

1. Sin `animation_group` y sin `range_x`/`range_y`.
2. Con `animation_group` pero sin rangos fijos.
3. Con las dos cosas.

Reprodúcelas y describe **qué se ve mal** en cada una de las dos primeras. Después
contesta: para un informe escrito, ¿usarías la animación o dos paneles? Justifícalo.

In [ ]:
# TODO: Escribe tu código aquí

### Ejercicio 4 (Intermedio): Medir para decidir

Un cliente te pide un gráfico de dispersión de un conjunto de **200.000 filas** para
publicarlo en su intranet.

1. Mide el HTML con `go.Scatter`, con `go.Scattergl` y con `px.density_heatmap`.
2. Mide el PNG equivalente con Matplotlib.
3. Abre los tres HTML en el navegador y comprueba cuál se puede mover con soltura.
4. Escribe la recomendación en cinco líneas, **citando tus números**, y di qué
   información se pierde con la opción que recomiendas.

In [ ]:
# TODO: Escribe tu código aquí

### Ejercicio 5 (Intermedio): Cuadro de mando de TechStore

Monta con `make_subplots` un cuadro de mando de cuatro paneles que responda a estas
cuatro preguntas, una por panel:

1. ¿Cómo evoluciona el importe mes a mes?
2. ¿Qué categorías concentran el gasto?
3. ¿Cómo se reparte el importe por transacción?
4. ¿Hay diferencias entre regiones?

Requisitos: `hover` personalizado en los cuatro, leyenda horizontal arriba, paleta
coherente, y **el título de cada panel formulado como la conclusión**, no como el
nombre de las variables.

Expórtalo a HTML autocontenido y di lo que pesa.

In [ ]:
# TODO: Escribe tu código aquí

### Ejercicio 6 (Avanzado): El coste de publicar los datos

La sección 1 decía que publicar un gráfico de Plotly es publicar los datos. Pruébalo.

1. Haz un gráfico con `px.scatter` de una muestra de `ventas` que incluya
   `hover_data=["Cliente_ID", "Producto"]`.
2. Genera el HTML con `to_html`.
3. **Busca dentro de la cadena** el identificador de un cliente concreto y demuestra
   que está ahí.
4. Escribe en tres líneas qué implicaciones tiene eso si el conjunto de datos
   contuviera nombres, correos o direcciones, y qué harías antes de publicar.

Este ejercicio no es sobre Plotly: es sobre lo que se entrega sin darse cuenta.

In [ ]:
# TODO: Escribe tu código aquí
# Pista: html = fig.to_html(); "C0098" in html

### Ejercicio 7 (Avanzado): La tabla de decisión, tuya

Rehaz la tabla de la sección 10 **con tus propias mediciones**, hechas sobre los datos
de TechStore. Para cada una de las nueve filas:

- reproduce la medición que la sostiene, o diseña una si no la hay,
- anota el número que obtienes,
- y marca las filas en las que tu medición **no** coincide con lo que dice la tabla.

Esa última parte es la importante: si algo no te sale, o la tabla está mal o tu
medición está mal, y averiguar cuál de las dos es el ejercicio.

Esta tabla, con tus números, es la base de la parte 5 de la práctica P4.1.

In [ ]:
# TODO: Escribe tu código aquí

## Resumen

1. **Plotly produce HTML, no una imagen.** De ahí salen todas sus ventajas y todos
   sus inconvenientes.
2. **El fichero lleva los datos dentro.** Crece con el número de puntos, y publicar
   el gráfico es publicar los datos.
3. **No toques `pio.renderers.default`.** Plotly detecta el entorno; ponerlo a
   «browser» es lo que hace que los gráficos no salgan en el cuaderno.
4. **El `hover` es lo que más aporta y lo que menos se personaliza.** Rompe la
   disyuntiva entre gráfico limpio y gráfico detallado.
5. **Una animación necesita `animation_group` y rangos fijos.** Sin lo segundo, lo
   que se mueve son los ejes.
6. **Para comparar dos momentos, paneles, no animación.** El ojo no compara con la
   memoria de hace tres segundos.
7. **El 3D interactivo sí gana** al estático, porque resuelve su única objeción: que
   el lector pueda girarlo.
8. **`Scattergl` por encima de cinco mil puntos**, y agregar por encima de medio
   millón. Es la misma conclusión del cuaderno 02 en otra biblioteca.
9. **`include_plotlyjs=True` para entregar**, aunque pese: un fichero que necesita
   internet para abrirse es un fichero que un día no se abre.
10. **El argumento del criterio 1.d es el ecosistema, no la velocidad.** NumPy,
    Pandas, Matplotlib y Seaborn se apilan y comparten estructuras de datos. Eso es
    lo que no tiene ningún lenguaje más rápido.

## Para seguir

- [Plotly Python](https://plotly.com/python/) — la documentación está organizada por
  tipo de gráfico, que es como se busca en la práctica.
- [Referencia completa de las figuras](https://plotly.com/python/reference/) — el
  sitio donde mirar qué opciones tiene un `trace`.
- [Dash](https://dash.plotly.com/) y [Streamlit](https://streamlit.io/) — el paso
  siguiente, cuando el cuadro de mando tiene que ser una aplicación. Streamlit ya se
  usó en la UD2.
- Hans Rosling, *200 Countries, 200 Years, 4 Minutes* (BBC, 2010). Cuatro minutos, y
  es la mejor clase de visualización de datos que hay grabada.

**Siguiente:** el cuaderno 05 deja de lado la sintaxis y va al criterio 2.e: cómo se
mira el resultado de un modelo para decidir si funciona.